# 07 — Inventory Initial Refactor

Verify the removal of `df_inventory_ts` from the public mock surface.
The mock now exposes a long-format `inventory_initial` DataFrame directly
(stations + depots, both commodities), and `DataLoaderGraph` reads it as
is — no more digging into the first row of an hourly MultiIndex matrix.
The full hourly matrix still exists internally inside the mock, but only
to seed the GBFS-like telemetry; it is never exposed.

In [ ]:
from gbp.loaders import DataLoaderMock

mock = DataLoaderMock({"n_stations": 5, "n_depots": 2, "n_timestamps": 24})
mock.load_data()

mock_surface = {
    "has_inventory_initial": hasattr(mock, "inventory_initial") and mock.inventory_initial is not None,
    "has_df_inventory_ts": hasattr(mock, "df_inventory_ts"),
    "inventory_initial_columns": list(mock.inventory_initial.columns),
    "inventory_initial_rows": len(mock.inventory_initial),
    "telemetry_rows": len(mock.df_telemetry_ts),
    "trip_rows": len(mock.df_trips),
}
mock_surface


In [ ]:
mock_inventory_initial = mock.inventory_initial.copy()
mock_inventory_initial


In [ ]:
from gbp.loaders import DataLoaderGraph, GraphLoaderConfig

loader = DataLoaderGraph(mock, GraphLoaderConfig())
raw = loader.load()

raw_inventory_initial = raw.inventory_initial.copy()
passthrough = {
    "raw_rows": len(raw_inventory_initial),
    "matches_source": len(
        raw_inventory_initial.merge(
            mock_inventory_initial,
            on=["facility_id", "commodity_category"],
            suffixes=("_raw", "_src"),
        ).query("quantity_raw == quantity_src")
    ) == len(raw_inventory_initial),
}
passthrough
